In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:

# load in dataframe with photic zone variable (using 4.5% threshold)
df3 = pd.read_csv("../../data/merged_dataset_organized_45photic.csv")
print(df3.columns)

Index(['ORD_OCC', 'CAST_ID', 'DATE_TIME_UTC', 'DATE_TIME_PST', 'LAT_DEC',
       'LON_DEC', 'STA_ID', 'LINE', 'STA', 'DEPTH', 'PRESSURE',
       'ESTCHL_CRUISECORR', 'ESTCHL_STACORR', 'BAT', 'XMISS', 'SPAR', 'PAR',
       'CHL_A', 'PHAEO', 'CAST_COUNT', 'Cruise_ID', 'Cruz_Sta', 'Cast_ID',
       'Sta_ID', 'Distance', 'Date', 'Time', 'Lat_Dec', 'Lon_Dec', 'Ac_Line',
       'Bottom_D', 'Secchi', 'IntChl', 'IntC14', 'TimeZone', 'Visibility',
       'PHOTIC_ZONE'],
      dtype='object')


Per 10-fold Cross Validation analysis: the photic depth threshold of 4.7% is the most viable option for calculating the photic zone depth

In [4]:
def calculate_photic_depth_47(group):
    """Calculate depth where PAR drops to 4.7% of its value at 2m baseline."""
    group = group.sort_values('DEPTH').dropna(subset=['PAR'])
    
    depth_2m = group[group['DEPTH'] == 2.0]
    if len(depth_2m) == 0 or len(group) < 2:
        return np.nan
    
    surface_par = depth_2m['PAR'].iloc[0]
    if surface_par <= 0:
        return np.nan
    
    group = group[group['DEPTH'] >= 2.0].copy()
    group['PAR_pct'] = (group['PAR'] / surface_par) * 100
    
    above = group[group['PAR_pct'] >= 4.7]
    below = group[group['PAR_pct'] < 4.7]
    
    if len(above) == 0 or len(below) == 0:
        return np.nan
    
    d_a, d_b = above['DEPTH'].iloc[-1], below['DEPTH'].iloc[0]
    p_a, p_b = above['PAR_pct'].iloc[-1], below['PAR_pct'].iloc[0]
    
    return d_a + (d_b - d_a) * (p_a - 4.7) / (p_a - p_b)

# Drop old 4.5% photic zone column
df3 = df3.drop(columns=['PHOTIC_ZONE'])

# Calculate photic depth (4.7% threshold) per cast
photic_zone = df3.groupby('CAST_COUNT').apply(calculate_photic_depth_47, include_groups=False)
photic_zone.name = 'photic_zone'

# Merge back into df3
df3 = df3.merge(photic_zone.reset_index(), on='CAST_COUNT', how='left')

print(f"Casts with valid photic_zone: {df3.groupby('CAST_COUNT')['photic_zone'].first().notna().sum()}")
print(df3.groupby('CAST_COUNT')['photic_zone'].first().describe())

Casts with valid photic_zone: 1971
count    1971.000000
mean       39.542413
std        20.313460
min         3.906000
25%        24.036411
50%        36.344262
75%        51.958200
max       136.805062
Name: photic_zone, dtype: float64


The new `PHOTIC_ZONE` column contains continuous photic depth values repeated across all CTD cast depths rather than a boolean value.

In [6]:
# write new dataset with 4.7% photic threshold as a parquet: data_47photic.parquet
df3.to_parquet('../Data/data_47photic.parquet', index=False)
print(f"Saved to Data/data_47photic.parquet ({len(df3)} rows)")

Saved to Data/data_47photic.parquet (1255966 rows)
